# 07 - Model Explainability and ML Lineage

This notebook demonstrates two critical MLOps capabilities:
1. **Explainability**: SHAP-based feature importance for individual predictions
2. **ML Lineage**: End-to-end tracing from source data through features to models

**Why it matters**:
- Regulatory compliance (explain why a customer was flagged)
- Model debugging (understand prediction drivers)
- Data governance (trace data flow through the ML pipeline)

**Prerequisites**: Run `02_model_training_xgboost.ipynb` first.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()
print(f"Connected: {session.get_current_user()}")

In [ ]:
# Load model from registry
reg = Registry(
    session=session,
    database_name="MLOPS_DEMO_DB",
    schema_name="PIPELINES"
)
mv = reg.get_model("CHURN_PREDICTION_MODEL").default
print(f"Loaded: {mv.model_name} version {mv.version_name}")

## Part 1: SHAP Explainability

SHAP (SHapley Additive exPlanations) values show each feature's contribution to a prediction.

- **Positive SHAP value**: Feature pushes prediction toward churn
- **Negative SHAP value**: Feature pushes prediction away from churn
- **Magnitude**: How strongly the feature influences the prediction

Built into the Model Registry -- no additional packages required.

In [ ]:
# Prepare sample data for explanation
explain_df = session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS").select(
    "CREDIT_SCORE", "AGE", "TENURE", "BALANCE",
    "NUM_OF_PRODUCTS", "HAS_CR_CARD", "IS_ACTIVE_MEMBER", "ESTIMATED_SALARY"
).limit(100)

print(f"Sample size for explanations: {explain_df.count()} rows")

In [ ]:
# Run SHAP explanations -- built-in, no extra packages
explanations = mv.run(explain_df, function_name="explain")
print("SHAP Explanations:")
explanations.show(5)

In [ ]:
# Aggregate SHAP values for global feature importance
import pandas as pd

shap_pd = explanations.to_pandas()

# SHAP output columns typically have a suffix
shap_cols = [c for c in shap_pd.columns if c not in [
    "CREDIT_SCORE", "AGE", "TENURE", "BALANCE",
    "NUM_OF_PRODUCTS", "HAS_CR_CARD", "IS_ACTIVE_MEMBER", "ESTIMATED_SALARY"
]]

if shap_cols:
    # Mean absolute SHAP value per feature = global importance
    importance = shap_pd[shap_cols].abs().mean().sort_values(ascending=False)
    print("Global Feature Importance (mean |SHAP|):")
    for feat, val in importance.items():
        print(f"  {feat}: {val:.4f}")
else:
    print("SHAP columns:")
    print(shap_pd.columns.tolist())

## Part 2: ML Lineage

ML Lineage traces the complete data flow:

```
Source Tables --> Feature Views --> Datasets --> Models --> Services
```

Lineage is captured automatically when:
- Features are registered in the Feature Store
- Models are logged with `sample_input_data`
- Models are deployed as services

Queryable via Python API, SQL, and visible in the SnowSight UI.

In [ ]:
# Query model lineage via Python API
try:
    lineage = mv.lineage()
    print("Model Lineage (upstream dependencies):")
    for item in lineage:
        item_type = type(item).__name__
        item_name = getattr(item, 'name', str(item))
        print(f"  [{item_type}] {item_name}")
except Exception as e:
    print(f"Lineage API: {e}")
    print("Lineage is visible in the SnowSight UI even if the API call varies by version.")

In [ ]:
# Query lineage via SQL (ACCESS_HISTORY based)
try:
    session.sql("""
    SELECT 
        DIRECTOBJECTNAME AS OBJECT_NAME,
        DIRECTOBJECTTYPE AS OBJECT_TYPE,
        ACCESSTYPE,
        QUERY_START_TIME
    FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY
    WHERE DIRECTOBJECTNAME LIKE '%CHURN_PREDICTION%'
       OR DIRECTOBJECTNAME LIKE '%CUSTOMER%'
    ORDER BY QUERY_START_TIME DESC
    LIMIT 20
    """).show()
except Exception as e:
    print(f"Note: ACCESS_HISTORY requires ACCOUNTADMIN or appropriate grants: {e}")

In [ ]:
# Feature Store lineage (if Feature Store was used)
try:
    from snowflake.ml.feature_store import FeatureStore, CreationMode
    
    fs = FeatureStore(
        session=session,
        database="MLOPS_DEMO_DB",
        name="FEATURES",
        default_warehouse="MLOPS_DEMO_WH",
        creation_mode=CreationMode.CREATE_IF_NOT_EXIST
    )
    
    fv_list = fs.list_feature_views().to_pandas()
    print("Feature Views (upstream of model):")
    print(fv_list[["NAME", "VERSION", "DESC"]].to_string(index=False))
except Exception as e:
    print(f"Feature Store lineage: {e}")

## What to Show in SnowSight

### Lineage Graph
Navigate to **AI/ML > Model Registry > CHURN_PREDICTION_MODEL > v1 > Lineage** to see:
- Visual graph showing data flow from source tables to the model
- Click on nodes to see details
- Upstream: which tables and features feed the model
- Downstream: which services and outputs consume the model

### Feature Store Lineage
Navigate to **AI/ML > Feature Store > CUSTOMER_FEATURES** to see:
- Which source tables feed the feature view
- Which models consume these features

**Key message**: Lineage is automatic and visual -- no manual documentation or external tools needed. Auditors can trace any prediction back to its source data.

---

**Next**: `08_model_serving.ipynb` - Deploy as REST API on SPCS